# 08 TensorFlow Lite Model Conversion

## Objective
Convert trained model to TensorFlow Lite for mobile deployment:
- ✓ Load best trained model
- ✓ Convert to TFLite format
- ✓ Quantize model for mobile efficiency
- ✓ Test TFLite model accuracy
- ✓ Compare model sizes
- ✓ Save deployment-ready model

**Prerequisites:** All previous notebooks (01-07)

### Import Libraries

import sys
import os

sys.path.insert(0, '../utils')

from model_utils import load_model

import numpy as np
import json
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd

print("✓ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

### Load Data and Model

print("Loading test data and model...\n")

# Load test data
X_test = np.load('../results/preprocessed_data/X_test.npy')
y_test = np.load('../results/preprocessed_data/y_test.npy')

# Load label encoding
with open('../results/preprocessed_data/label_encoding.json', 'r') as f:
    label_encoding = json.load(f)

num_classes = len(label_encoding)

# Load best model
with open('../results/reports/best_model.json', 'r') as f:
    best_model_info = json.load(f)

best_model_name = best_model_info['name']

if 'mobilenet' in best_model_name.lower():
    model_path = '../models/mobilenet/mobilenet_v2.h5'
else:
    model_path = '../models/cnn/cnn_baseline.h5'

model = load_model(model_path)

print(f"✓ Model loaded: {best_model_name}")
print(f"✓ Test data shape: {X_test.shape}")

### 1. Convert Model to TFLite (Float)

print("\n" + "="*70)
print("CONVERTING TO TFLITE FORMAT")
print("="*70 + "\n")

print("Step 1: Converting to TFLite (Float)...\n")

# Create directory
os.makedirs('../models/tflite', exist_ok=True)

# Convert
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model_float = converter.convert()

# Save float model
float_model_path = '../models/tflite/model_float.tflite'
with open(float_model_path, 'wb') as f:
    f.write(tflite_model_float)

print(f"✓ Converted to TFLite (Float)")
print(f"  Saved: {float_model_path}")

### 2. Convert Model to TFLite (Quantized)

print("\nStep 2: Converting to TFLite (Quantized)...\n")

# Create quantization dataset
def representative_data_gen():
    # Use first 100 samples
    for i in range(min(100, len(X_test))):
        yield [X_test[i:i+1].astype(np.float32)]

# Convert with quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_data_gen = representative_data_gen
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

tflite_model_quantized = converter.convert()

# Save quantized model
quant_model_path = '../models/tflite/model_quantized.tflite'
with open(quant_model_path, 'wb') as f:
    f.write(tflite_model_quantized)

print(f"✓ Converted to TFLite (Quantized)")
print(f"  Saved: {quant_model_path}")

### 3. Model Size Comparison

print("\n" + "="*70)
print("MODEL SIZE COMPARISON")
print("="*70 + "\n")

# Get file sizes
original_size = os.path.getsize(model_path) / (1024 * 1024)  # MB
float_size = os.path.getsize(float_model_path) / (1024 * 1024)  # MB
quant_size = os.path.getsize(quant_model_path) / (1024 * 1024)  # MB

print(f"Original Model (.h5): {original_size:.2f} MB")
print(f"TFLite Float Model: {float_size:.2f} MB ({float_size/original_size*100:.1f}%)")
print(f"TFLite Quantized Model: {quant_size:.2f} MB ({quant_size/original_size*100:.1f}%)")

print(f"\nSize Reduction:")
print(f"  Float vs Original: {(1 - float_size/original_size)*100:.1f}%")
print(f"  Quantized vs Original: {(1 - quant_size/original_size)*100:.1f}%")
print(f"  Quantized vs Float: {(1 - quant_size/float_size)*100:.1f}%")

### 4. Plot Model Size Comparison

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
models = ['Original\n(H5)', 'TFLite\n(Float)', 'TFLite\n(Quantized)']
sizes = [original_size, float_size, quant_size]
colors = ['#3498db', '#2ecc71', '#e74c3c']

bars = ax1.bar(models, sizes, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Size (MB)', fontweight='bold', fontsize=12)
ax1.set_title('Model Size Comparison', fontweight='bold', fontsize=12)
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bar, size in zip(bars, sizes):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{size:.2f} MB',
            ha='center', va='bottom', fontweight='bold')

# Pie chart
reduction_data = [quant_size, original_size - quant_size]
ax2.pie(reduction_data, labels=['Quantized Size', 'Size Reduced'],
        autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'],
        startangle=90, textprops={'fontweight': 'bold'})
ax2.set_title(f'Size Reduction: {(1 - quant_size/original_size)*100:.1f}%',
             fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('../results/plots/model_size_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: model_size_comparison.png")

### 5. Test TFLite Model Accuracy

print("\n" + "="*70)
print("TESTING TFLITE MODEL ACCURACY")
print("="*70 + "\n")

def evaluate_tflite(tflite_path, X_test, y_test, batch_size=32):
    """Evaluate TFLite model"""
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    predictions = []
    
    for i in range(0, len(X_test), batch_size):
        batch = X_test[i:i+batch_size].astype(np.float32)
        
        for img in batch:
            interpreter.set_tensor(input_details[0]['index'], np.expand_dims(img, axis=0))
            interpreter.invoke()
            output = interpreter.get_tensor(output_details[0]['index'])
            predictions.append(output[0])
    
    predictions = np.array(predictions)
    pred_labels = np.argmax(predictions, axis=1)
    true_labels = np.argmax(y_test, axis=1)
    
    accuracy = np.mean(pred_labels == true_labels)
    return accuracy

print("Evaluating Float TFLite model...")
float_accuracy = evaluate_tflite(float_model_path, X_test, y_test)
print(f"✓ Float TFLite Accuracy: {float_accuracy:.4f}")

print("\nEvaluating Quantized TFLite model...")
quant_accuracy = evaluate_tflite(quant_model_path, X_test, y_test)
print(f"✓ Quantized TFLite Accuracy: {quant_accuracy:.4f}")

### 6. Accuracy Comparison

print("\n" + "="*70)
print("ACCURACY COMPARISON")
print("="*70 + "\n")

# Original model accuracy
original_pred = model.predict(X_test, verbose=0)
original_accuracy = np.mean(np.argmax(original_pred, axis=1) == np.argmax(y_test, axis=1))

accuracies = {
    'Original Model': original_accuracy,
    'TFLite Float': float_accuracy,
    'TFLite Quantized': quant_accuracy
}

comparison_df = pd.DataFrame([
    {'Model': k, 'Accuracy': v, 'Accuracy_Drop': 0 if k == 'Original Model' else original_accuracy - v}
    for k, v in accuracies.items()
])

print(comparison_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

models = list(accuracies.keys())
accs = list(accuracies.values())
colors = ['#3498db', '#2ecc71', '#e74c3c']

bars = ax.bar(models, accs, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Accuracy', fontweight='bold', fontsize=12)
ax.set_title('Model Accuracy Comparison', fontweight='bold', fontsize=14)
ax.set_ylim([0, 1.0])
ax.grid(axis='y', alpha=0.3)

for bar, acc in zip(bars, accs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{acc:.4f}',
           ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/plots/tflite_accuracy_comparison.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: tflite_accuracy_comparison.png")

### 7. Generate Conversion Report

print("\n" + "="*70)
print("GENERATING CONVERSION REPORT")
print("="*70 + "\n")

conversion_report = {
    'model': best_model_name,
    'original_model': {
        'path': model_path,
        'size_mb': float(original_size),
        'accuracy': float(original_accuracy)
    },
    'tflite_float': {
        'path': float_model_path,
        'size_mb': float(float_size),
        'accuracy': float(float_accuracy),
        'size_reduction_percent': float((1 - float_size/original_size)*100)
    },
    'tflite_quantized': {
        'path': quant_model_path,
        'size_mb': float(quant_size),
        'accuracy': float(quant_accuracy),
        'size_reduction_percent': float((1 - quant_size/original_size)*100),
        'accuracy_drop_percent': float((original_accuracy - quant_accuracy)*100)
    },
    'deployment_recommendation': 'TFLite Quantized' if quant_accuracy > 0.95 * original_accuracy else 'TFLite Float'
}

with open('../results/reports/tflite_conversion_report.json', 'w') as f:
    json.dump(conversion_report, f, indent=4)

print("✓ Conversion report saved")
print(json.dumps(conversion_report, indent=2))

### 8. Create Mobile-Ready Model Metadata

print("\nCreating mobile model metadata...\n")

# Load label encoding
with open('../results/preprocessed_data/label_encoding.json', 'r') as f:
    label_encoding = json.load(f)

mobile_metadata = {
    'model_name': best_model_name,
    'model_type': 'Image Classification',
    'input_shape': [224, 224, 3],
    'input_normalization': 'Normalize to [0, 1] by dividing by 255',
    'num_classes': num_classes,
    'classes': label_encoding,
    'tflite_model': 'model_quantized.tflite',
    'version': '1.0',
    'inference_time_ms_estimate': '50-100 ms',
    'memory_requirements_mb': float(quant_size)
}

with open('../models/tflite/metadata.json', 'w') as f:
    json.dump(mobile_metadata, f, indent=4)

print("✓ Mobile metadata created: ../models/tflite/metadata.json")

### 9. Summary

print("\n" + "="*70)
print("✓ TFLITE CONVERSION COMPLETE")
print("="*70)

print(f"\n📊 Conversion Summary:")
print(f"  Original Model: {original_size:.2f} MB ({original_accuracy:.4f} accuracy)")
print(f"  TFLite Float: {float_size:.2f} MB ({float_accuracy:.4f} accuracy)")
print(f"  TFLite Quantized: {quant_size:.2f} MB ({quant_accuracy:.4f} accuracy)")
print(f"\n  Size Reduction: {(1 - quant_size/original_size)*100:.1f}%")
print(f"  Accuracy Maintained: {(original_accuracy - quant_accuracy)*100:.2f}% drop")

print(f"\n💾 Files Saved:")
print(f"  - ../models/tflite/model_float.tflite")
print(f"  - ../models/tflite/model_quantized.tflite")
print(f"  - ../models/tflite/metadata.json")
print(f"  - ../results/reports/tflite_conversion_report.json")

print(f"\n🚀 Deployment Ready: ../models/tflite/model_quantized.tflite")
print(f"   Use this model for mobile/edge deployment")

print(f"\n→ Next Step: 09_gradcam_explainability.ipynb")